# 06 — Building an Animated MP4 from Scratch

Turn a sequence of forecast frames into an MP4 video using `imageio` — no `noaawc`.

The pattern is:
1. Load all forecast steps into an xarray Dataset
2. Render each time step as a PNG frame (skip if it already exists)
3. Encode all frames into an MP4

---
**Stack used:** `noawclg`, `numpy`, `matplotlib`, `cartopy`, `cmocean`, `imageio`

In [ ]:
import noawclg
import numpy as np
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — required for saving without display
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import imageio
import copy
import os
from pathlib import Path

In [ ]:
BG = "#0d1117"; LAND = "#13171d"; OCEAN_C = "#0d1520"
COAST = "#2d6db5"; BORDER = "#2a2f36"
TEXT = "#e6edf3"; SUBTEXT = "#8b949e"
BOXBG = "#161b22"; BOXEDGE = "#30363d"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    BG,
    "savefig.facecolor": BG,
    "text.color":        TEXT,
    "font.family":       "monospace",
})

## 1. Load GFS data — all forecast hours

In [ ]:
ds = noawclg.load(
    var="t2m",
    run_date="20260418",
    cycle="00",
    forecast_hours=range(0, 121, 3),   # 41 frames
)

lat = ds["lat"].values
lon = ds["lon"].values

# Precompute lon conversion once
lon_180 = np.where(lon > 180, lon - 360, lon)
order   = np.argsort(lon_180)
lon_180 = lon_180[order]

n_frames = len(ds["t2m"].time)
print(f"{n_frames} frames loaded")

## 2. Define the colormap and levels once

In [ ]:
levels = np.arange(-20, 52, 2)
cmap   = copy.copy(cmocean.cm.thermal)
cmap.set_under(alpha=0); cmap.set_bad(alpha=0)
norm   = mcolors.BoundaryNorm(levels, ncolors=cmap.N, clip=False)

## 3. Render each frame to PNG

Frames are cached — if the PNG already exists, we skip it.  
This lets you restart an interrupted run without re-rendering everything.

In [ ]:
FRAMES_DIR = Path("frames_t2m_ortho")
FRAMES_DIR.mkdir(exist_ok=True)

# Camera: static globe centred on South America
CENTRAL_LON = -50.0
CENTRAL_LAT = -15.0

proj = ccrs.Orthographic(CENTRAL_LON, CENTRAL_LAT)


def render_frame(tidx: int, fpath: Path) -> None:
    """Render one time step to a PNG file."""
    field = ds["t2m"].isel(time=tidx).values - 273.15
    field = field[:, order]
    time_val = str(ds["t2m"].time.values[tidx])[:16].replace("T", " ")

    fig = plt.figure(figsize=(10, 10), facecolor=BG)
    ax  = fig.add_subplot(1, 1, 1, projection=proj)
    ax.set_facecolor(BG)
    ax.set_global()

    ax.add_feature(cfeature.OCEAN.with_scale("110m"),  facecolor=OCEAN_C, zorder=0)
    ax.add_feature(cfeature.LAND.with_scale("110m"),   facecolor=LAND,    zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("110m"),
                   edgecolor=COAST, linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS.with_scale("110m"),
                   edgecolor=BORDER, linewidth=0.4, zorder=3)

    cf = ax.pcolormesh(lon_180, lat, field, cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(), zorder=1)

    ax.contour(lon_180[::3], lat[::3], field[::3, ::3],
               levels=levels[::5], colors="white",
               linewidths=0.2, alpha=0.35,
               transform=ccrs.PlateCarree(), zorder=2)

    cb = fig.colorbar(cf, ax=ax, orientation="horizontal",
                      pad=0.03, fraction=0.03, shrink=0.85)
    cb.set_label("2m Temperature (°C)", fontsize=8, color=SUBTEXT)
    cb.ax.tick_params(labelsize=7, colors=SUBTEXT)
    cb.outline.set_edgecolor(BOXEDGE)

    ax.set_title("2m Temperature", fontsize=10, fontweight="bold",
                 color=TEXT, loc="left", pad=6)
    ax.set_title(f"GFS 00z  {time_val}", fontsize=7,
                 color=SUBTEXT, loc="right", pad=6)

    fig.tight_layout()
    fig.savefig(fpath, dpi=120, bbox_inches="tight")
    plt.close(fig)


for tidx in range(n_frames):
    fpath = FRAMES_DIR / f"frame_{tidx:04d}.png"
    if not fpath.exists():
        render_frame(tidx, fpath)
    print(f"  frame {tidx + 1}/{n_frames}  →  {fpath}", end="\r")

print(f"\nAll {n_frames} frames ready in {FRAMES_DIR}/")

## 4. Encode frames into MP4

In [ ]:
OUTPUT_FILE = "t2m_ortho.mp4"
FPS         = 10

frame_paths = sorted(FRAMES_DIR.glob("frame_*.png"))

with imageio.get_writer(OUTPUT_FILE, fps=FPS, codec="libx264", quality=8) as writer:
    for fpath in frame_paths:
        writer.append_data(imageio.imread(fpath))

print(f"Saved: {OUTPUT_FILE}  ({len(frame_paths)} frames @ {FPS} fps)")

## 5. Camera rotation during animation

To smoothly pan the camera, compute a `(lon, lat)` position per frame and
rebuild the axes with a new projection each time.

In [ ]:
def camera_arc(
    tidx: int,
    n_frames: int,
    lon_start: float, lon_end: float,
    lat_start: float, lat_end: float,
    stop_fraction: float = 0.7,
) -> tuple[float, float]:
    """Linear interpolation along a camera arc; freezes at stop_fraction."""
    stop = max(1, int(round(stop_fraction * n_frames)))
    if tidx >= stop:
        return lon_end, lat_end
    t = tidx / stop
    lon = lon_start + t * (lon_end - lon_start)
    lat = lat_start + t * (lat_end - lat_start)
    return lon, lat


# Example: arc from (-90, -5) → (-20, -20), freeze at 70% of frames
FRAMES_DIR2 = Path("frames_t2m_rotating")
FRAMES_DIR2.mkdir(exist_ok=True)

for tidx in range(n_frames):
    fpath = FRAMES_DIR2 / f"frame_{tidx:04d}.png"
    if fpath.exists():
        print(f"  frame {tidx + 1}/{n_frames}  (cached)", end="\r")
        continue

    cam_lon, cam_lat = camera_arc(
        tidx, n_frames,
        lon_start=-90, lon_end=-20,
        lat_start=-5,  lat_end=-20,
        stop_fraction=0.7,
    )

    field = ds["t2m"].isel(time=tidx).values - 273.15
    field = field[:, order]
    time_val = str(ds["t2m"].time.values[tidx])[:16].replace("T", " ")

    proj_r = ccrs.Orthographic(cam_lon, cam_lat)
    fig = plt.figure(figsize=(10, 10), facecolor=BG)
    ax  = fig.add_subplot(1, 1, 1, projection=proj_r)
    ax.set_facecolor(BG)
    ax.set_global()

    ax.add_feature(cfeature.OCEAN.with_scale("110m"),  facecolor=OCEAN_C, zorder=0)
    ax.add_feature(cfeature.LAND.with_scale("110m"),   facecolor=LAND,    zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("110m"),
                   edgecolor=COAST, linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS.with_scale("110m"),
                   edgecolor=BORDER, linewidth=0.4, zorder=3)

    cf = ax.pcolormesh(lon_180, lat, field, cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(), zorder=1)

    cb = fig.colorbar(cf, ax=ax, orientation="horizontal",
                      pad=0.03, fraction=0.03, shrink=0.85)
    cb.set_label("2m Temperature (°C)", fontsize=8, color=SUBTEXT)
    cb.ax.tick_params(labelsize=7, colors=SUBTEXT)
    cb.outline.set_edgecolor(BOXEDGE)

    ax.set_title("2m Temperature", fontsize=10, fontweight="bold",
                 color=TEXT, loc="left", pad=6)
    ax.set_title(f"GFS 00z  {time_val}", fontsize=7,
                 color=SUBTEXT, loc="right", pad=6)

    fig.tight_layout()
    fig.savefig(fpath, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  frame {tidx + 1}/{n_frames}", end="\r")

print(f"\nRendered {n_frames} rotating frames.")

In [ ]:
OUTPUT_ROT = "t2m_rotating.mp4"
frame_paths2 = sorted(FRAMES_DIR2.glob("frame_*.png"))

with imageio.get_writer(OUTPUT_ROT, fps=10, codec="libx264", quality=8) as writer:
    for fpath in frame_paths2:
        writer.append_data(imageio.imread(fpath))

print(f"Saved: {OUTPUT_ROT}")

## 6. Exercises

1. **GIF output** — change `OUTPUT_FILE` to `"t2m.gif"` and remove the `codec` argument from `imageio.get_writer()`.
2. **PlateCarrée animation** — adapt `render_frame()` to use `ccrs.PlateCarree()` and `ax.set_extent()` instead of `set_global()`.
3. **H.265 encoding** — change `codec="libx265"` and add `output_params=["-pix_fmt", "yuv420p"]` for ~40% smaller files.
4. **Speed up** — use `multiprocessing.Pool` to render frames in parallel (only safe if each worker creates its own `Figure`).
5. **Nearside animation** — add a `camera_arc()` call that sweeps `SAT_LON` from west to east while the forecast runs.